# Three-Model Comparison: Baseline (I²) vs GPR(I²) vs GPR(cbrt)  G1M1

This notebook compares three versions of the electrolyzer degradation model on the G1M1 dataset:
1. **Baseline (I2)**: Original Urc1 model using I² nonlinear term
2. **GPR Smoothed (I2)**: GPR-smoothed version of the Baseline model
3. **GPR Smoothed (cbrt)**: Urc1_cbrt_GPR model using cube-root nonlinear term (I-c6)^(1/3)

Performance is compared across three reference conditions (Low / Medium / High).


In [1]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from pathlib import Path
from datetime import datetime

# Project imports
from degradation_toolbox.Urc.Urc1 import Urc1
from degradation_toolbox.Urc.Urc1_gpr import Urc1_GPR
from degradation_toolbox.Urc.Urc1_cbrt_gpr import Urc1_cbrt_GPR
from master_arbeit_Di.explore.UnifiedModelComparator import UnifiedModelComparator
from master_arbeit_Di.explore.GMpreprocess import GMpreprocess

print(" All imports successful")
print(f"Current directory: {os.getcwd()}")

 All imports successful
Current directory: c:\Users\Z0057NPT\Documents\MA_code\master_arbeit_Di\GPR


In [2]:
now = datetime.now()
formatted_time = now.strftime("%Y-%m-%d %H:%M:%S")
print("=" * 80)
print("Training start time:", formatted_time)
print("=" * 80)

Training start time: 2026-05-04 16:44:12


In [3]:
# =============================================================================

# CONFIGURATION SECTION - EDIT THESE TO CUSTOMIZE YOUR ANALYSIS

# =============================================================================



# Dataset and directories

DATASET_PATH = r"..\\..\\explore_data\\G1M1_new.parquet"

PREPROCESS_OUTPUT_DIR = r"..\\..\\explore_data\\output"

PLOTS_OUTPUT_DIR = r"..\\plots\\GPR_cbrt\\G1M1_comparison"



# Create plots directory if it doesn't exist

os.makedirs(PLOTS_OUTPUT_DIR, exist_ok=True)



# Reference condition configurations: Low, Medium, High

REF_CONFIGS = {

    "Low": {

        "Iref": 0.3,

        "Tref": 58,

        "OHref": 11,

        "gt_file": r"..\\ground_truth\\output_backup\\gt_raw_processed\\G1M1_new__gt_diagram__g1m1_low_load__iref_0p3__tref_58__ohref_11__daily_regression_full_coverage.csv",

    },

    "Medium": {

        "Iref": 1.0,

        "Tref": 58,

        "OHref": 100,

        "gt_file": r"..\\ground_truth\\output_backup\\gt_raw_processed\\G1M1_new__gt_diagram__g1m1_mid_load__iref_1__tref_58__ohref_100__daily_regression_full_coverage.csv",

    },

    "High": {

        "Iref": 1.48,

        "Tref": 57,

        "OHref": 100,

        "gt_file": r"..\\ground_truth\\output_backup\\gt_raw_processed\\G1M1_new__gt_diagram__g1m1_high_load__iref_1p48__tref_57__ohref_100__daily_regression_full_coverage.csv",

    },

}



# Model fitting common config (shared across all models)

COMMON_CONFIG = {

    "Iref": [cfg["Iref"] for cfg in REF_CONFIGS.values()],

    "Tref": 60,

    "OHref": 72,

    "ref_config": REF_CONFIGS,

    "len_interval": 2,

    "slide": 1,

    "min_num_data_required_for_fit": 300,

    "threshold": 1e6,

    "i_off": 0.1,

    "u_off": 1.3,

    "plot_fit": 0,

    "data_filter_i_min": 0.1,

    "data_filter_U_min": 1.4,

    "data_filter_U_max": 2.3,

    "data_filter_T_min": 50,

    "data_filter_T_max": 65,

}



# GPR-specific parameters (shared for GPR models)

GPR_PARAMS = {

    "gpr_r2_threshold": 0.8,

    "gpr_cond_threshold": 1e6,

    "length_scale_init": 500 * 24,  # 500 days in hours

    "output_mode": "all",  # "all" or "filtered"

}



# Three model variants to run

MODEL_VARIANTS = ["I2", "gpr_i2", "gpr_cbrt"]



# Comparison settings

SHOW_GT_METRICS = True

SHOW_ALL_COND_METRICS = True



print("Ã¢Å“â€œ Configuration loaded successfully")

print(f"  - Reference conditions: {list(REF_CONFIGS.keys())}")

print(f"  - Model variants: {MODEL_VARIANTS}")

print(f"  - Output directory: {PLOTS_OUTPUT_DIR}")


Ã¢Å“â€œ Configuration loaded successfully
  - Reference conditions: ['Low', 'Medium', 'High']
  - Model variants: ['I2', 'gpr_i2', 'gpr_cbrt']
  - Output directory: ..\\plots\\GPR_cbrt\\G1M1_comparison


In [4]:
# =============================================================================
# STEP 1: DATA LOADING & PREPROCESSING
# =============================================================================
print("=" * 80)
print("STEP 1: Data Loading & Preprocessing")
print("=" * 80)

preprocessor = GMpreprocess(file_path=DATASET_PATH, output_dir=PREPROCESS_OUTPUT_DIR)
data = preprocessor.run()
dataset_name = preprocessor.name

print(f"\nDataset: {dataset_name}")
print(f"  - Shape: {data.shape}")
print(f"  - Time range: {data.index.min()} Ã¢â€ â€™ {data.index.max()}")

# Preprocess once and reuse across all three models
shared_pre = Urc1.preprocess_once(
    data,
    i_off=COMMON_CONFIG["i_off"],
    u_off=COMMON_CONFIG["u_off"],
    data_filter_i_min=COMMON_CONFIG["data_filter_i_min"],
    data_filter_U_min=COMMON_CONFIG["data_filter_U_min"],
    data_filter_U_max=COMMON_CONFIG["data_filter_U_max"],
    data_filter_T_min=COMMON_CONFIG["data_filter_T_min"],
    data_filter_T_max=COMMON_CONFIG["data_filter_T_max"],
)
print(f"\n Shared preprocessed data: {len(shared_pre)} rows")

STEP 1: Data Loading & Preprocessing
>> [Init] Created output directory: ..\\..\\explore_data\\output
=== 1. Loading & Preprocessing: G1M1_new ===
>> Data loaded successfully.
   [Detected] Temperature Col: 'Temp_Module_1' -> ID: '1'
>> Columns renamed to standard format.
>> Columns filtered. Retained: ['currentDensity', 'temperature', 'voltage']

=== 3. Saving Preprocessd data in .parquet format ===
>> ✅ Final Results saved successfully to:
   ..\\..\\explore_data\\output\G1M1_new_20260504_164413.parquet

=== GMpreprocess Pipeline Completed Successfully ===

Dataset: G1M1_new
  - Shape: (2529217, 3)
  - Time range: 2021-02-12 00:00:00 Ã¢â€ â€™ 2025-12-09 09:59:00
[preprocess_once] 2529217 -> 1069062 points.

 Shared preprocessed data: 1069062 rows


In [5]:
# =============================================================================
# STEP 2: TRAIN THREE MODELS (Baseline I2, GPR I2, GPR cbrt)
# =============================================================================
print("\n" + "=" * 80)
print("STEP 2: Model Training (Baseline I2 | GPR I2 | GPR cbrt)")
print("=" * 80)

models = {}
model_training_times = {}

for variant in MODEL_VARIANTS:
    print(f"\n  --> Training {variant.upper()} model...")
    t_start = datetime.now()
    
    try:
        if variant == "I2":
            # Baseline Urc1 with I2 term
            model = Urc1(
                data=data,
                name=dataset_name,
                preprocessed_data=shared_pre,
                **COMMON_CONFIG,
            )
            models["Baseline (I2)"] = model
            print(f"    Training time: {(datetime.now() - t_start).total_seconds():.1f}s")
            model_training_times["Baseline (I2)"] = (datetime.now() - t_start).total_seconds()
            
        elif variant == "gpr_i2":
            # GPR-smoothed Urc1 with I2 term
            model = Urc1_GPR(
                data=data,
                name=dataset_name,
                preprocessed_data=shared_pre,
                **COMMON_CONFIG,
                **GPR_PARAMS,
            )
            models["GPR Smoothed (I2)"] = model
            print(f"    Training time: {(datetime.now() - t_start).total_seconds():.1f}s")
            model_training_times["GPR Smoothed (I2)"] = (datetime.now() - t_start).total_seconds()
            
        elif variant == "gpr_cbrt":
            # NEW: GPR-smoothed Urc1 with cbrt term
            model = Urc1_cbrt_GPR(
                data=data,
                name=dataset_name,
                preprocessed_data=shared_pre,
                **COMMON_CONFIG,
                **GPR_PARAMS,
            )
            models["GPR Smoothed (cbrt)"] = model
            print(f"     Training time: {(datetime.now() - t_start).total_seconds():.1f}s")
            model_training_times["GPR Smoothed (cbrt)"] = (datetime.now() - t_start).total_seconds()
            
    except Exception as e:
        print(f"     Training failed: {e}")

print(f"\n--> Successfully trained {len(models)}/3 models")
print(f"  - Model names: {list(models.keys())}")
print(f"  - Total training time: {sum(model_training_times.values()):.1f}s")


STEP 2: Model Training (Baseline I2 | GPR I2 | GPR cbrt)

  --> Training I2 model...
Using shared preprocessed data (1069062 points, skipping preprocess).
Voltage model fitting ...
Fitting Stats: 474 intervals low data, 0 fit failed.
1098 out of 1766 fitting results are reliable.
    Training time: 16.4s

  --> Training GPR_I2 model...
Using shared preprocessed data (1069062 points, skipping preprocess).
Voltage model fitting ...
Fitting Stats: 474 intervals low data, 0 fit failed.
1098 out of 1766 fitting results are reliable.


c:\Users\Z0057NPT\Documents\MA_code\venv\Lib\site-packages\sklearn\gaussian_process\kernels.py:452: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified upper bound 0.2. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
c:\Users\Z0057NPT\Documents\MA_code\venv\Lib\site-packages\sklearn\gaussian_process\kernels.py:442: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__k2__length_scale is close to the specified lower bound 2400.0. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(
c:\Users\Z0057NPT\Documents\MA_code\venv\Lib\site-packages\sklearn\gaussian_process\kernels.py:452: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified upper bound 0.2. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
c:\Users\Z0057NPT\Documents\MA_code\venv\

    Training time: 236.1s

  --> Training GPR_CBRT model...
Using shared preprocessed data (1069062 points, skipping preprocess).
Voltage model fitting (nonlinear_term='cbrt') ...
Fitting Stats: 474 intervals low data, 8 fit failed.


C:\Users\Z0057NPT\Documents\MA_code\degradation_toolbox\Urc\Urc1_c6_sigmoid.py:951: RuntimeWarning: invalid value encountered in sqrt
  se = np.where(variance > 0, np.sqrt(variance) * scale, np.nan)
C:\Users\Z0057NPT\Documents\MA_code\degradation_toolbox\Urc\Urc1_c6_sigmoid.py:951: RuntimeWarning: invalid value encountered in sqrt
  se = np.where(variance > 0, np.sqrt(variance) * scale, np.nan)


1042 out of 1766 fitting results are reliable.


c:\Users\Z0057NPT\Documents\MA_code\venv\Lib\site-packages\sklearn\gaussian_process\kernels.py:442: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__k2__length_scale is close to the specified lower bound 2400.0. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(
c:\Users\Z0057NPT\Documents\MA_code\venv\Lib\site-packages\sklearn\gaussian_process\kernels.py:452: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified upper bound 0.2. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
c:\Users\Z0057NPT\Documents\MA_code\venv\Lib\site-packages\sklearn\gaussian_process\kernels.py:452: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified upper bound 0.2. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
c:\Users\Z0057NPT\Documents\MA_code\venv\

     Training time: 551.5s

--> Successfully trained 3/3 models
  - Model names: ['Baseline (I2)', 'GPR Smoothed (I2)', 'GPR Smoothed (cbrt)']
  - Total training time: 803.9s


In [6]:
now = datetime.now()
formatted_time = now.strftime("%Y-%m-%d %H:%M:%S")
print("\n" + "=" * 80)
print("Training finished time:", formatted_time)
print("=" * 80)


Training finished time: 2026-05-04 16:58:26


In [7]:
# =============================================================================
# STEP 3: COMPARATOR SETUP & GROUND TRUTH LOADING
# =============================================================================
print("\n" + "=" * 80)
print("STEP 3: Initialize Comparator & Load Ground Truth")
print("=" * 80)

comparator = UnifiedModelComparator(models)
print(f" UnifiedModelComparator initialized with {len(models)} models")

# Determine project root
try:
    project_root = Path(__file__).parent.parent
except NameError:
    project_root = Path.cwd()

print(f"  Project root: {project_root}\n")

# Load GT for each reference config
gt_loaded_count = 0
for ref_name, ref_cfg in REF_CONFIGS.items():
    iref = ref_cfg["Iref"]
    gt_file = ref_cfg["gt_file"]
    
    # Resolve path
    gt_path = Path(gt_file)
    if not gt_path.is_absolute():
        gt_path = project_root / gt_path
    
    gt_path = gt_path.resolve()
    
    print(f"  Looking for GT file: {gt_path}")
    
    if gt_path.exists():
        try:
            gt_data = pd.read_csv(gt_path, index_col=0, parse_dates=True)
            
            # Normalize column names
            colmap = {str(c).strip().lower(): c for c in gt_data.columns}
            
            # Priority order for GT voltage columns
            candidate_cols = [
                "gt_uref_regression",
                "voltage",
                "uref",
                "gt_uref",
            ]
            
            selected_col = None
            for c in candidate_cols:
                if c in colmap:
                    selected_col = colmap[c]
                    break
            
            if selected_col is not None:
                gt_series = gt_data[selected_col]
            elif len(gt_data.columns) == 1:
                gt_series = gt_data.iloc[:, 0]
                selected_col = gt_data.columns[0]
            else:
                raise ValueError(f"Cannot identify voltage column in {gt_path}")
            
            gt_series = gt_series.dropna()
            
            # Set as GT for this Iref
            comparator.set_ground_truth(gt_series, iref=iref)
            gt_loaded_count += 1
            print(f"  Loaded GT for {ref_name} (Iref={iref}): {len(gt_series)} points [column: {selected_col}]")
            
        except Exception as e:
            print(f"  Failed to load GT for {ref_name}: {e}")
    else:
        print(f"  GT file NOT found: {gt_path}")

has_gt = gt_loaded_count > 0
print(f"\n{'='*80}")
print(f"GT status: {gt_loaded_count}/{len(REF_CONFIGS)} reference conditions loaded")
print(f"GT metrics will be {'ENABLED Ã¢Å“â€œ' if has_gt else 'DISABLED (no GT files found)'}")
print(f"{'='*80}\n")


STEP 3: Initialize Comparator & Load Ground Truth
 UnifiedModelComparator initialized with 3 models
  Project root: c:\Users\Z0057NPT\Documents\MA_code\master_arbeit_Di\GPR

  Looking for GT file: C:\Users\Z0057NPT\Documents\MA_code\master_arbeit_Di\ground_truth\output_backup\gt_raw_processed\G1M1_new__gt_diagram__g1m1_low_load__iref_0p3__tref_58__ohref_11__daily_regression_full_coverage.csv
  Loaded GT for Low (Iref=0.3): 1762 points [column: gt_uref_regression]
  Looking for GT file: C:\Users\Z0057NPT\Documents\MA_code\master_arbeit_Di\ground_truth\output_backup\gt_raw_processed\G1M1_new__gt_diagram__g1m1_mid_load__iref_1__tref_58__ohref_100__daily_regression_full_coverage.csv
  Loaded GT for Medium (Iref=1.0): 1762 points [column: gt_uref_regression]
  Looking for GT file: C:\Users\Z0057NPT\Documents\MA_code\master_arbeit_Di\ground_truth\output_backup\gt_raw_processed\G1M1_new__gt_diagram__g1m1_high_load__iref_1p48__tref_57__ohref_100__daily_regression_full_coverage.csv
  Loaded GT

In [8]:
# =============================================================================

# STEP 4: REFERENCE-SPECIFIC ANALYSIS & METRICS

# =============================================================================

print("=" * 80)

print("STEP 4: Reference-Specific Metrics & Comparison")

print("=" * 80)



rate_tables = []



for ref_name, ref_cfg in REF_CONFIGS.items():

    iref = ref_cfg["Iref"]

    tref = ref_cfg["Tref"]

    ohref = ref_cfg["OHref"]

    

    print(f"\n REFERENCE CONDITION: {ref_name} (Iref={iref}, Tref={tref}, OHref={ohref}) Ã¢â€“â€œÃ¢â€“â€œÃ¢â€“â€œ")

    

    # --- Metrics Table ---

    df_metrics = comparator.compare_all(

        i_target=iref,

        include_all_cond_metrics=SHOW_ALL_COND_METRICS,

        include_gt_metrics=has_gt,

    )

    print(f"\nPerformance Metrics:")

    print(df_metrics.to_markdown(index=False))

    

    rate_tables.append(

        df_metrics[["Model Name", "Target Current (A/cm2)", "Degradation Rate (uV/h)", "Slope Sigma (uV/h)"]].assign(

            Reference=ref_name

        )

    )

    

    # --- Trend Plot ---

    try:

        comparator.plot_interactive_trends(

            target_i=iref,

            show_gt=has_gt,

            save=True,

            output_dir=PLOTS_OUTPUT_DIR,

            uncertainty_style="band",

            uncertainty_opacity=0.12,

            show_series_line=True,

            rate_precision=6,

        )

        print(f" Trend plot saved for {ref_name}")

    except Exception as e:

        print(f" Trend plot failed for {ref_name}: {e}")

    

    # --- Detailed Report ---

    print(f"\nDetailed Comparison Report:")

    comparator.print_comparison_report(

        i_target=iref,

        include_all_cond_metrics=SHOW_ALL_COND_METRICS,

        include_gt_metrics=has_gt,

    )


STEP 4: Reference-Specific Metrics & Comparison

 REFERENCE CONDITION: Low (Iref=0.3, Tref=58, OHref=11) Ã¢â€“â€œÃ¢â€“â€œÃ¢â€“â€œ

Performance Metrics:
| Model Name          |   Target Current (A/cm2) |   Fitting Time (s) |   Data Points (n) |   Degradation Rate (uV/h) |   RMSE (mV) |   Slope Sigma (uV/h) |   Mono (Rank) [0-1] |   Outlier Count |   Outlier (%) |   Max Residual (mV) |   Mean SE (mV) |   Cond Median log10 |   Cond P95 log10 |   Cond Count (Used Scope) | Cond Scope   |   GT RMSE (mV) |   GT MAE (mV) |   GT Valid Intervals |
|:--------------------|-------------------------:|-------------------:|------------------:|--------------------------:|------------:|---------------------:|--------------------:|----------------:|--------------:|--------------------:|---------------:|--------------------:|-----------------:|--------------------------:|:-------------|---------------:|--------------:|---------------------:|
| Baseline (I2)       |                      0.3 |             1

In [9]:
# This cell is intentionally left blank (duplicate removed)
pass


In [10]:
# =============================================================================
# STEP 5: CROSS-MODEL DIAGNOSTICS
# =============================================================================
print("\n" + "=" * 80)
print("STEP 5: Cross-Model Diagnostics")
print("=" * 80)

# Fit quality (RMSE & R2 distributions)
try:
    print("\n Plotting fit quality (RMSE & R² distributions)...")
    comparator.plot_fit_quality(save=True, output_dir=PLOTS_OUTPUT_DIR)
    print(" Fit quality plot saved")
except Exception as e:
    print(f" Fit quality plot failed: {e}")

# Coefficient diagnostics (including c6 for cbrt model)
try:
    print("\n Plotting coefficient diagnostics...")
    comparator.plot_coefficient_diagnostic(
        include_c6=True,
        save=True,
        output_dir=PLOTS_OUTPUT_DIR
    )
    print(" Coefficient diagnostic plot saved (c6 included for cbrt model)")
except Exception as e:
    print(f" Coefficient diagnostic plot failed: {e}")

# Coverage analysis (Gantt chart of valid intervals across all Irefs)
try:
    print("\n Plotting coverage Gantt (all Irefs)...")
    comparator.plot_coverage_gantt(save=True, output_dir=PLOTS_OUTPUT_DIR)
    print("Coverage Gantt saved")
except Exception as e:
    print(f" Coverage Gantt failed: {e}")


STEP 5: Cross-Model Diagnostics

 Plotting fit quality (RMSE & R² distributions)...
 Fit quality plot saved

 Plotting coefficient diagnostics...
 Coefficient diagnostic plot saved (c6 included for cbrt model)

 Plotting coverage Gantt (all Irefs)...
Coverage Gantt saved


In [11]:
# =============================================================================

# STEP 6: CUSTOM REFERENCE VOLTAGE EXTRACTION

# =============================================================================

print("\n" + "=" * 80)

print("STEP 6: Custom Reference Voltage Extraction (calculate_urc_for_custom_refs)")

print("=" * 80)



# Build ref_list from REF_CONFIGS

ref_list = []

for ref_name, ref_cfg in REF_CONFIGS.items():

    ref_list.append({

        "Iref": ref_cfg["Iref"],

        "Tref": ref_cfg["Tref"],

        "OHref": ref_cfg["OHref"],

        "name": ref_name,

    })



# For each model, calculate custom Urc values

for model_name, model in models.items():

    print(f"\n Calculating custom references for {model_name}...")

    try:

        urc_dict = model.calculate_urc_for_custom_refs(ref_list)

        print(f" Extracted Urc for {len(urc_dict)} reference configurations:")

        for ref_name, urc_df in urc_dict.items():

            print(f"   - {ref_name}: {len(urc_df)} time points")

    except Exception as e:

        print(f" Failed: {e}")



STEP 6: Custom Reference Voltage Extraction (calculate_urc_for_custom_refs)

 Calculating custom references for Baseline (I2)...
Calculated Urc for Low: 1098 points
Calculated Urc for Medium: 1098 points
Calculated Urc for High: 1098 points
 Extracted Urc for 3 reference configurations:
   - Low: 1098 time points
   - Medium: 1098 time points
   - High: 1098 time points

 Calculating custom references for GPR Smoothed (I2)...
Calculated Urc for Low: 1766 points
Calculated Urc for Medium: 1766 points
Calculated Urc for High: 1766 points
 Extracted Urc for 3 reference configurations:
   - Low: 1766 time points
   - Medium: 1766 time points
   - High: 1766 time points

 Calculating custom references for GPR Smoothed (cbrt)...
Calculated Urc for Low: 1766 points
Calculated Urc for Medium: 1766 points
Calculated Urc for High: 1766 points
 Extracted Urc for 3 reference configurations:
   - Low: 1766 time points
   - Medium: 1766 time points
   - High: 1766 time points


In [12]:
# =============================================================================
# SUMMARY
# =============================================================================
print("\n" + "=" * 80)
print("ANALYSIS COMPLETE")
print("=" * 80)
print(f"\n Trained models: {len(models)}")
print(f"  - {list(models.keys())}")
print(f"\n Reference conditions analyzed: {len(REF_CONFIGS)}")
print(f"  - {list(REF_CONFIGS.keys())}")
print(f"\n Ground truth data: {'Loaded' if has_gt else 'Not available'}")
print(f" Output plots saved to: {PLOTS_OUTPUT_DIR}/")
print(f"\nKey settings:")
print(f"  - All condition metrics: {SHOW_ALL_COND_METRICS}")
print(f"  - GT metrics included: {SHOW_GT_METRICS}")
print(f"  - GPR length scale: {GPR_PARAMS['length_scale_init']/(24)} days")
print(f"  - GPR output mode: {GPR_PARAMS['output_mode']}")
print(f"\nModel-specific notes:")
print(f"  1. Baseline (I2): Original Urc1 with 5 parameters (c1-c5)")
print(f"  2. GPR Smoothed (I2): Baseline with Gaussian Process smoothing")
print(f"  3. GPR Smoothed (cbrt): NEW model with 6 parameters (c1-c6), cbrt nonlinear term")
print(f"\nTo modify analysis: Edit DATASET_PATH, REF_CONFIGS, COMMON_CONFIG, and MODEL_VARIANTS at the top.")


ANALYSIS COMPLETE

 Trained models: 3
  - ['Baseline (I2)', 'GPR Smoothed (I2)', 'GPR Smoothed (cbrt)']

 Reference conditions analyzed: 3
  - ['Low', 'Medium', 'High']

 Ground truth data: Loaded
 Output plots saved to: ..\\plots\\GPR_cbrt\\G1M1_comparison/

Key settings:
  - All condition metrics: True
  - GT metrics included: True
  - GPR length scale: 500.0 days
  - GPR output mode: all

Model-specific notes:
  1. Baseline (I2): Original Urc1 with 5 parameters (c1-c5)
  2. GPR Smoothed (I2): Baseline with Gaussian Process smoothing
  3. GPR Smoothed (cbrt): NEW model with 6 parameters (c1-c6), cbrt nonlinear term

To modify analysis: Edit DATASET_PATH, REF_CONFIGS, COMMON_CONFIG, and MODEL_VARIANTS at the top.
